In [ ]:
"""
Mortality Prediction with Chronos-2 Embeddings
Optimized: Sample windows first, then generate embeddings
"""

import scipy.io
import torch
import numpy as np
import pandas as pd
from chronos import Chronos2Pipeline
import pickle as pl
import glob
import traceback
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torchvision.ops import sigmoid_focal_loss

In [ ]:
# ============================================================================
# CONSTANTS
# ============================================================================

CONTEXT_SIZE_HOURS = 80
CONTEXT_SIZE_SECONDS = CONTEXT_SIZE_HOURS * 3600  # 288,000 seconds
WINDOW_SIZE_SECONDS = 300  # 5 minutes
NUM_WINDOWS = CONTEXT_SIZE_SECONDS // WINDOW_SIZE_SECONDS  # 960 windows

# Thresholds for prediction
THRESHOLDS_DAYS = [3, 5, 7]
THRESHOLDS_SECONDS = [d * 86400 for d in THRESHOLDS_DAYS]

# Overlap configuration
OVERLAP_PERCENTAGE_NON_NICU = 0.0   # No overlap
OVERLAP_PERCENTAGE_NICU = 0.90      # 90% overlap
NUM_WINDOWS_PER_BABY = 20

# Sampling configuration
NUM_LABEL_0_NICU = 10
NUM_LABEL_1_NICU = 10
NUM_LABEL_0_NON_NICU = 10

In [ ]:
# ============================================================================
# HELPER FUNCTIONS (UNCHANGED)
# ============================================================================

def extract_raw_timeseries(mat_file_path, start_time, context_size):
    """Extract raw HR and SPO2 time series data from MATLAB file"""
    try:
        mat_data = scipy.io.loadmat(mat_file_path)
    except Exception as e:
        print(f"Error loading {mat_file_path}: {e}")
        return None
    
    birth_time = mat_data['pdata'][0][1]
    age_at_death = mat_data['pdata'][0][-1]
    death_ts = age_at_death * 86400 if not np.isnan(age_at_death) else None
    
    hr_cols = mat_data['vdata'][:,0]
    spo2_cols = mat_data['vdata'][:,2]
    timestamps = mat_data['vt'].flatten()
    
    end_time = start_time + context_size
    time_mask = (timestamps >= start_time) & (timestamps < end_time)
    valid_mask = ~np.isnan(hr_cols) & ~np.isnan(spo2_cols)
    final_mask = time_mask & valid_mask
    
    timestamps_valid = timestamps[final_mask]
    hr_valid = hr_cols[final_mask]
    spo2_valid = spo2_cols[final_mask]
    
    if len(timestamps_valid) == 0:
        return None
    
    return {
        'timestamps': timestamps_valid,
        'hr_raw': hr_valid,
        'spo2_raw': spo2_valid,
        'birth_time': birth_time,
        'death_ts': death_ts,
        'start_time': start_time,
        'context_size': context_size
    }

def create_windowed_features_from_raw(raw_data, window_size):
    """Convert raw time series into fixed-size windowed features"""
    timestamps = raw_data['timestamps']
    hr_raw = raw_data['hr_raw']
    spo2_raw = raw_data['spo2_raw']
    start_time = raw_data['start_time']
    context_size = raw_data['context_size']
    
    window_indices = np.floor((timestamps - start_time) / window_size).astype(int)
    num_windows = int(np.ceil(context_size / window_size))
    
    hr_windows = np.full(num_windows, np.nan)
    spo2_windows = np.full(num_windows, np.nan)
    
    for idx in range(num_windows):
        mask = window_indices == idx
        if np.any(mask):
            hr_windows[idx] = np.mean(hr_raw[mask])
            spo2_windows[idx] = np.mean(spo2_raw[mask])
    
    return hr_windows, spo2_windows

def create_windowed_features(mat_file_path, start_time, context_size, window_size):
    """Convenience function: combines raw extraction + windowing"""
    raw_data = extract_raw_timeseries(mat_file_path, start_time, context_size)
    if raw_data is None:
        return None
    return create_windowed_features_from_raw(raw_data, window_size)

def prepare_chronos_input(hr_windows, spo2_windows):
    """Prepare windowed data for Chronos model input"""
    hr_clean = np.where(
        np.isnan(hr_windows), 
        np.nanmean(hr_windows) if not np.all(np.isnan(hr_windows)) else 0, 
        hr_windows
    ).astype(np.float32)
    
    spo2_clean = np.where(
        np.isnan(spo2_windows), 
        np.nanmean(spo2_windows) if not np.all(np.isnan(spo2_windows)) else 0, 
        spo2_windows
    ).astype(np.float32)
    
    hr_tensor = torch.tensor(hr_clean, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    spo2_tensor = torch.tensor(spo2_clean, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    
    context = torch.cat([hr_tensor, spo2_tensor], dim=0)
    return context

def get_chronos_embedding(chronos_input, pipeline):
    """Generate embedding from Chronos model"""
    with torch.no_grad():
        embeddings, _ = pipeline.embed(chronos_input)
    
    hr_emb = embeddings[0][0][-1].float()
    spo2_emb = embeddings[1][0][-1].float()
    combined_embedding = torch.cat([hr_emb, spo2_emb], dim=0).cpu().numpy()
    
    return combined_embedding

def get_embedding(hr_windows, spo2_windows, pipeline):
    """Convenience function: combines preparation + embedding generation"""
    chronos_input = prepare_chronos_input(hr_windows, spo2_windows)
    return get_chronos_embedding(chronos_input, pipeline)


def create_window_key(mat_file, window_start):
    """Create unique key: baby_id_timestamp"""
    baby_id = mat_file.split('/')[-1].replace('.mat', '')
    return f"{baby_id}_{int(window_start)}"


def get_or_generate_embedding(mat_file, window_start, pipeline, cache):
    """Get from cache or generate new"""
    key = create_window_key(mat_file, window_start)
    
    if key in cache:
        return cache[key], True  # From cache
    
    # Generate new
    try:
        result = create_windowed_features(mat_file, window_start, 
                                         CONTEXT_SIZE_SECONDS, WINDOW_SIZE_SECONDS)
        if result is not None:
            embedding = get_embedding(result[0], result[1], pipeline)
            cache[key] = embedding
            return embedding, False  # Newly generated
    except:
        pass
    
    return None, False


In [ ]:
# ============================================================================
# STEP 1: GENERATE WINDOW METADATA (THRESHOLD-INDEPENDENT)
# ============================================================================

def generate_window_metadata_for_baby(mat_file_path, overlap_percentage, 
                                     max_windows=NUM_WINDOWS_PER_BABY):
    """
    Generate ALL possible windows for a single baby.
    
    Constraints (NO threshold considerations):
        - birth_time: Start of life
        - death_time or last_timestamp: End of available data
        - window_size: CONTEXT_SIZE_SECONDS (80 hours)
        - overlap_percentage: How much adjacent windows overlap
    """
    try:
        mat_data = scipy.io.loadmat(mat_file_path)
    except Exception as e:
        return pd.DataFrame()
    
    birth_time = mat_data['pdata'][0][1]
    age_at_death = mat_data['pdata'][0][-1]
    death_ts = age_at_death * 86400 if not np.isnan(age_at_death) else np.nan
    
    if np.isnan(birth_time):
        return pd.DataFrame()
    
    timestamps = mat_data['vt'].flatten()
    last_ts = timestamps[-1].item()
    
    baby_type = 'NICU' if not np.isnan(death_ts) else 'NON_NICU'
    baby_id = mat_file_path.split('/')[-1].replace('.mat', '')
    
    available_time = last_ts - birth_time
    
    if available_time < CONTEXT_SIZE_SECONDS:
        return pd.DataFrame()
    
    # Calculate step size based on overlap
    step_size = CONTEXT_SIZE_SECONDS * (1 - overlap_percentage)
    
    num_possible_windows = int((available_time - CONTEXT_SIZE_SECONDS) / step_size) + 1
    num_windows = min(num_possible_windows, max_windows)
    
    if num_windows <= 0:
        num_windows = 1
        step_size = 0
    
    # Generate windows
    windows = []
    for i in range(num_windows):
        window_start = birth_time + i * step_size
        window_end = window_start + CONTEXT_SIZE_SECONDS
        
        if window_end <= last_ts:
            windows.append({
                'mat_file': mat_file_path,
                'baby_id': baby_id,
                'window_idx': i,
                'window_start': window_start,
                'window_end': window_end,
                'baby_type': baby_type,
                'death_time': death_ts,
                'birth_time': birth_time
            })
    
    return pd.DataFrame(windows)


def generate_all_window_metadata(nicu_files, non_nicu_files):
    """Generate window metadata for ALL babies"""
    print("\n" + "="*80)
    print(" "*20 + "STEP 1: GENERATING WINDOW METADATA")
    print(" "*15 + "(Threshold-Independent, Physical Constraints Only)")
    print("="*80 + "\n")
    
    all_metadata = []
    
    # Non-NICU babies (no overlap)
    print(f"Processing {len(non_nicu_files)} Non-NICU babies (no overlap)...")
    for i, file_path in enumerate(non_nicu_files):
        baby_id = file_path.split('/')[-1].replace('.mat', '')
        
        if (i + 1) % 20 == 0 or i + 1 == len(non_nicu_files):
            print(f"  Progress: {i+1}/{len(non_nicu_files)} babies")
        
        try:
            df = generate_window_metadata_for_baby(
                file_path, 
                overlap_percentage=OVERLAP_PERCENTAGE_NON_NICU,
                max_windows=NUM_WINDOWS_PER_BABY
            )
            all_metadata.append(df)
        except Exception as e:
            print(f"  ✗ Error with baby {baby_id}: {str(e)}")
    
    # NICU babies (90% overlap)
    print(f"\nProcessing {len(nicu_files)} NICU babies (90% overlap)...")
    for i, file_path in enumerate(nicu_files):
        baby_id = file_path.split('/')[-1].replace('.mat', '')
        
        if (i + 1) % 10 == 0 or i + 1 == len(nicu_files):
            print(f"  Progress: {i+1}/{len(nicu_files)} babies")
        
        try:
            df = generate_window_metadata_for_baby(
                file_path, 
                overlap_percentage=OVERLAP_PERCENTAGE_NICU,
                max_windows=NUM_WINDOWS_PER_BABY
            )
            all_metadata.append(df)
        except Exception as e:
            print(f"  ✗ Error with baby {baby_id}: {str(e)}")
    
    # Combine all metadata
    metadata_df = pd.concat(all_metadata, ignore_index=True)
    
    print(f"\n{'='*80}")
    print(f"✓ Total windows generated: {len(metadata_df)}")
    print(f"  Non-NICU: {len(metadata_df[metadata_df['baby_type'] == 'NON_NICU'])}")
    print(f"  NICU: {len(metadata_df[metadata_df['baby_type'] == 'NICU'])}")
    print(f"{'='*80}\n")
    
    # Save metadata
    metadata_df.to_pickle('window_metadata.pkl')
    print("✓ Saved: window_metadata.pkl\n")
    
    return metadata_df


In [ ]:
# ============================================================================
# STEP 2: ASSIGN LABELS & SAMPLE WINDOWS FOR ALL THRESHOLDS
# ============================================================================

def assign_labels_for_threshold(metadata_df, threshold_seconds):
    """Assign labels based on threshold"""
    df = metadata_df.copy()
    
    # Non-NICU: Always label 0
    df.loc[df['baby_type'] == 'NON_NICU', 'label'] = 0
    
    # NICU: Label based on threshold
    nicu_mask = df['baby_type'] == 'NICU'
    threshold_start = df.loc[nicu_mask, 'death_time'] - threshold_seconds
    
    df.loc[nicu_mask, 'label'] = (
        df.loc[nicu_mask, 'window_end'] > threshold_start
    ).astype(int)
    
    return df


def sample_windows_for_all_thresholds(metadata_df):
    """
    Sample windows for ALL thresholds at once.
    Returns unified DataFrame with window selections.
    """
    print("\n" + "="*80)
    print(" "*20 + "STEP 2: SAMPLING WINDOWS FOR ALL THRESHOLDS")
    print("="*80 + "\n")
    
    all_sampled_windows = []
    sampled_indices_set = set()  # Track which windows we've selected
    
    for threshold_days, threshold_seconds in zip(THRESHOLDS_DAYS, THRESHOLDS_SECONDS):
        print(f"Sampling for {threshold_days}-day threshold...")
        
        # Assign labels for this threshold
        df = assign_labels_for_threshold(metadata_df, threshold_seconds)
        
        sampled_for_threshold = []
        
        # Sample Non-NICU babies
        non_nicu_babies = df[df['baby_type'] == 'NON_NICU']['baby_id'].unique()
        for baby_id in non_nicu_babies:
            baby_windows = df[(df['baby_id'] == baby_id) & (df['label'] == 0)]
            n_sample = min(NUM_LABEL_0_NON_NICU, len(baby_windows))
            
            if n_sample > 0:
                indices = np.linspace(0, len(baby_windows)-1, n_sample, dtype=int)
                selected = baby_windows.iloc[indices].copy()
                selected['threshold_days'] = threshold_days
                sampled_for_threshold.append(selected)
                sampled_indices_set.update(selected.index.tolist())
        
        # Sample NICU babies
        nicu_babies = df[df['baby_type'] == 'NICU']['baby_id'].unique()
        for baby_id in nicu_babies:
            baby_data = df[df['baby_id'] == baby_id]
            
            # Label 0 windows
            label_0_windows = baby_data[baby_data['label'] == 0]
            n_sample_0 = min(NUM_LABEL_0_NICU, len(label_0_windows))
            if n_sample_0 > 0:
                indices = np.linspace(0, len(label_0_windows)-1, n_sample_0, dtype=int)
                selected = label_0_windows.iloc[indices].copy()
                selected['threshold_days'] = threshold_days
                sampled_for_threshold.append(selected)
                sampled_indices_set.update(selected.index.tolist())
            
            # Label 1 windows
            label_1_windows = baby_data[baby_data['label'] == 1]
            n_sample_1 = min(NUM_LABEL_1_NICU, len(label_1_windows))
            if n_sample_1 > 0:
                indices = np.linspace(0, len(label_1_windows)-1, n_sample_1, dtype=int)
                selected = label_1_windows.iloc[indices].copy()
                selected['threshold_days'] = threshold_days
                sampled_for_threshold.append(selected)
                sampled_indices_set.update(selected.index.tolist())
        
        threshold_sampled = pd.concat(sampled_for_threshold, ignore_index=False)
        all_sampled_windows.append(threshold_sampled)
        
        print(f"  ✓ Sampled {len(threshold_sampled)} windows")
        print(f"    Label 0: {(threshold_sampled['label'] == 0).sum()}")
        print(f"    Label 1: {(threshold_sampled['label'] == 1).sum()}\n")
    
    # Combine all sampled windows
    all_sampled_df = pd.concat(all_sampled_windows, ignore_index=False)
    
    # Get unique windows that need embeddings
    unique_window_indices = sorted(list(sampled_indices_set))
    unique_windows_df = metadata_df.loc[unique_window_indices].copy()
    
    print(f"{'='*80}")
    print(f"✓ Total sampled window selections: {len(all_sampled_df)}")
    print(f"✓ Unique windows needing embeddings: {len(unique_windows_df)}")
    print(f"  (Same window may be used for multiple thresholds)")
    print(f"{'='*80}\n")
    
    # Save sampled selections
    all_sampled_df.to_pickle('sampled_windows_all_thresholds.pkl')
    print("✓ Saved: sampled_windows_all_thresholds.pkl")
    
    # Save unique windows that need embeddings
    unique_windows_df.to_pickle('windows_to_embed.pkl')
    print("✓ Saved: windows_to_embed.pkl\n")
    
    return all_sampled_df, unique_windows_df

In [ ]:
# ============================================================================
# STEP 3: GENERATE EMBEDDINGS FOR SAMPLED WINDOWS ONLY
# ============================================================================

def generate_embeddings_for_windows(windows_df, pipeline):
    """Generate embeddings with caching"""
    print("\n" + "="*80)
    print(" "*15 + "STEP 3: GENERATING EMBEDDINGS (WITH CACHING)")
    print("="*80 + "\n")
    
    # Load existing cache
    import os
    if os.path.exists('embeddings_global.pkl'):
        with open('embeddings_global.pkl', 'rb') as f:
            cache = pl.load(f)
        print(f"Loaded {len(cache)} cached embeddings")
    else:
        cache = {}
        print("Starting fresh cache")
    
    initial_size = len(cache)
    embeddings_dict = {}
    cache_hits = 0
    cache_misses = 0
    
    # Process windows
    for idx, row in windows_df.iterrows():
        embedding, was_cached = get_or_generate_embedding(
            row['mat_file'], row['window_start'], pipeline, cache
        )
        
        if embedding is not None:
            embeddings_dict[idx] = embedding
            if was_cached:
                cache_hits += 1
            else:
                cache_misses += 1
        
        # Progress
        if len(embeddings_dict) % 50 == 0:
            print(f"  {len(embeddings_dict)}/{len(windows_df)} | "
                  f"Cache: {cache_hits}, New: {cache_misses}")
    
    # Summary
    print(f"\n✓ Generated {len(embeddings_dict)} embeddings")
    print(f"  Cache hits: {cache_hits} ({cache_hits/len(windows_df)*100:.1f}%)")
    print(f"  New: {cache_misses}")
    print(f"  Global cache: {initial_size} → {len(cache)} "
          f"(+{len(cache)-initial_size})\n")
    
    # Save
    with open('embeddings_global.pkl', 'wb') as f:
        pl.dump(cache, f)
    with open('embeddings_index_mapping.pkl', 'wb') as f:
        pl.dump(embeddings_dict, f)
    
    print("✓ Saved: embeddings_global.pkl")
    print("✓ Saved: embeddings_index_mapping.pkl\n")
    
    return embeddings_dict


In [ ]:
# ============================================================================
# STEP 4: PREPARE DATA FOR TRAINING
# ============================================================================

def prepare_training_data_for_threshold(threshold_days, embeddings_dict=None):
    """Prepare data for one threshold"""
    print(f"\nPreparing {threshold_days}-day threshold...")
    
    if embeddings_dict is None:
        with open('embeddings_index_mapping.pkl', 'rb') as f:
            embeddings_dict = pl.load(f)
    
    sampled_all = pd.read_pickle('sampled_windows_all_thresholds.pkl')
    threshold_data = sampled_all[sampled_all['threshold_days'] == threshold_days]
    
    embeddings = []
    labels = []
    
    for idx, row in threshold_data.iterrows():
        if idx in embeddings_dict:
            embeddings.append(embeddings_dict[idx])
            labels.append(int(row['label']))
    
    print(f"  ✓ {len(embeddings)} embeddings (L0:{labels.count(0)}, L1:{labels.count(1)})")
    
    with open(f'embeddings_{threshold_days}d.pkl', 'wb') as f:
        pl.dump({'embeddings': embeddings, 'labels': labels}, f)
    
    return embeddings, labels


In [ ]:
# ============================================================================
# STEP 5: TRAIN CLASSIFIER (BINARY WITH SIGMOID FOCAL LOSS)
# ============================================================================

class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = [torch.FloatTensor(e) for e in embeddings]
        self.labels = torch.FloatTensor(labels)  # ← Float for binary
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


class Classifier(nn.Module):
    """Binary classifier - outputs 1 raw logit (NO sigmoid)"""
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden // 2, 1)  # ← 1 output for binary classification
        )
    
    def forward(self, x):
        return self.net(x.view(x.size(0), -1)).squeeze()  # ← Raw logits


def train_epoch(model, loader, optimizer, device, use_focal_loss=False, alpha=0.25, gamma=2.0):
    """Train one epoch"""
    model.train()
    total_loss = 0
    
    for embs, labels in loader:
        embs, labels = embs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        logits = model(embs)  # Raw logits
        
        if use_focal_loss:
            # sigmoid_focal_loss applies sigmoid internally
            loss = sigmoid_focal_loss(logits, labels, alpha=alpha, gamma=gamma, reduction='mean')
        else:
            # BCEWithLogitsLoss also applies sigmoid internally
            loss = nn.BCEWithLogitsLoss()(logits, labels)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    return total_loss / len(loader)


def validate(model, loader, device):
    """Validate model"""
    model.eval()
    preds, lbls, probs = [], [], []
    
    with torch.no_grad():
        for embs, labels in loader:
            embs, labels = embs.to(device), labels.to(device)
            logits = model(embs)  # Raw logits
            
            # Apply sigmoid for inference
            probs_batch = torch.sigmoid(logits).cpu().numpy()
            preds_batch = (probs_batch > 0.5).astype(int)
            
            probs.extend(probs_batch)
            preds.extend(preds_batch)
            lbls.extend(labels.cpu().numpy())
    
    return {
        'accuracy': accuracy_score(lbls, preds),
        'precision': precision_score(lbls, preds, zero_division=0),
        'recall': recall_score(lbls, preds, zero_division=0),
        'f1': f1_score(lbls, preds, zero_division=0),
        'auc': roc_auc_score(lbls, probs) if len(set(lbls)) > 1 else 0
    }


def k_fold_cv(embeddings, labels, k=5, epochs=50, batch_size=32, lr=0.001, 
              device='cpu', use_focal_loss=False, alpha=0.25, gamma=2.0):
    """K-Fold CV with optional focal loss"""
    if len(embeddings) == 0:
        print("No embeddings to train on!")
        return None, None, None
    
    embeddings = np.array(embeddings)
    labels = np.array(labels)
    dim = embeddings[0].flatten().shape[0]
    
    # Auto-compute alpha if None
    if use_focal_loss and alpha is None:
        n_neg = (labels == 0).sum()
        alpha = n_neg / len(labels)
    
    loss_type = f"Focal Loss (α={alpha:.3f}, γ={gamma})" if use_focal_loss else "Binary CE"
    print(f"\n{'='*80}")
    print(f"K-Fold CV with {loss_type}")
    print(f"Samples: {len(labels)}, Folds: {k}, Embedding dim: {dim}")
    print(f"Class distribution: Label 0: {(labels==0).sum()}, Label 1: {(labels==1).sum()}")
    print(f"{'='*80}\n")
    
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    fold_metrics = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(embeddings, labels)):
        print(f"Fold {fold+1}/{k}...", end=" ")
        
        train_data = EmbeddingDataset(embeddings[train_idx], labels[train_idx])
        val_data = EmbeddingDataset(embeddings[val_idx], labels[val_idx])
        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_data, batch_size=batch_size)
        
        model = Classifier(dim).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        
        for _ in range(epochs):
            train_epoch(model, train_loader, optimizer, device, 
                       use_focal_loss=use_focal_loss, alpha=alpha, gamma=gamma)
        
        metrics = validate(model, val_loader, device)
        fold_metrics.append(metrics)
        print(f"Acc: {metrics['accuracy']:.4f}, F1: {metrics['f1']:.4f}, "
              f"Recall: {metrics['recall']:.4f}, AUC: {metrics['auc']:.4f}")
    
    avg = {k: np.mean([f[k] for f in fold_metrics]) for k in fold_metrics[0]}
    std = {k: np.std([f[k] for f in fold_metrics]) for k in fold_metrics[0]}
    
    print(f"\n{'='*80}")
    print(f"AVERAGE METRICS ({loss_type})")
    print('─'*80)
    for k in avg:
        print(f"{k.capitalize():12s}: {avg[k]:.4f} ± {std[k]:.4f}")
    print('='*80)
    
    return avg, std, fold_metrics

In [ ]:
# ============================================================================
# COMPLETE PIPELINE
# ============================================================================

def run_complete_pipeline(use_focal_loss=False, gamma=2.0):
    """
    Complete pipeline with optional Focal Loss.
    
    Args:
        use_focal_loss: If True, use Focal Loss instead of Cross-Entropy
        gamma: Focal loss gamma parameter (default 2.0)
    """
    loss_name = f"Focal Loss (γ={gamma})" if use_focal_loss else "Cross-Entropy"
    print("\n" + "="*100)
    print(" "*15 + f"MORTALITY PREDICTION PIPELINE WITH {loss_name.upper()}")
    print("="*100 + "\n")
    
    # Load files
    with open('valid_summary_dic.pkl', 'rb') as f:
        valid_summary_dic = pl.load(f)
    
    nicu_mask = valid_summary_dic['indicator_vector'] == 1
    nicu_baby_ids = set(valid_summary_dic['baby_id'][nicu_mask])
    non_nicu_baby_ids = set(valid_summary_dic['baby_id'][~nicu_mask])
    
    all_mat_files = sorted(glob.glob('./download/Vitals/*.mat'))
    nicu_files = [f for f in all_mat_files if f.split('/')[-1] in nicu_baby_ids]
    non_nicu_files = [f for f in all_mat_files if f.split('/')[-1] in non_nicu_baby_ids]
    
    print(f"✓ {len(nicu_files)} NICU files")
    print(f"✓ {len(non_nicu_files)} Non-NICU files\n")
    
    # STEP 1: Metadata
    import os
    if os.path.exists('window_metadata.pkl'):
        metadata_df = pd.read_pickle('window_metadata.pkl')
        print(f"✓ Loaded metadata: {len(metadata_df)} windows\n")
    else:
        metadata_df = generate_all_window_metadata(nicu_files, non_nicu_files)
    
    # STEP 2: Sample windows
    if os.path.exists('windows_to_embed.pkl'):
        unique_windows_df = pd.read_pickle('windows_to_embed.pkl')
        print(f"✓ Loaded sampled windows: {len(unique_windows_df)}\n")
    else:
        _, unique_windows_df = sample_windows_for_all_thresholds(metadata_df)
    
    # STEP 3: Generate embeddings
    if os.path.exists('embeddings_index_mapping.pkl'):
        with open('embeddings_index_mapping.pkl', 'rb') as f:
            embeddings_dict = pl.load(f)
        print(f"✓ Loaded embeddings: {len(embeddings_dict)}\n")
    else:
        pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", 
                                                    device_map="cpu", 
                                                    dtype=torch.bfloat16)
        embeddings_dict = generate_embeddings_for_windows(unique_windows_df, pipeline)
    
    # STEP 4-5: Train each threshold
    for threshold_days in THRESHOLDS_DAYS:
        print("\n" + "="*100)
        print(f" "*30 + f"THRESHOLD: {threshold_days} DAYS ({loss_name})")
        print("="*100)
        
        embeddings, labels = prepare_training_data_for_threshold(threshold_days, embeddings_dict)
        
        # Train with chosen loss
        avg, std, fold_metrics = k_fold_cv(
            embeddings, labels, 
            k=5, epochs=50, batch_size=32, lr=0.001, device='cpu',
            use_focal_loss=use_focal_loss, gamma=gamma
        )
        
        # Save results with appropriate suffix
        suffix = 'focal' if use_focal_loss else 'ce'
        filename = f'results_{threshold_days}d_{suffix}.pkl'
        with open(filename, 'wb') as f:
            pl.dump({
                'avg': avg, 
                'std': std, 
                'fold_metrics': fold_metrics,
                'loss_type': loss_name,
                'gamma': gamma if use_focal_loss else None
            }, f)
        print(f"\n✓ Saved: {filename}")
    
    print("\n" + "="*100)
    print(" "*30 + "✓ PIPELINE COMPLETE")
    print("="*100 + "\n")

In [ ]:
run_complete_pipeline(use_focal_loss=False)
run_complete_pipeline(use_focal_loss=True, gamma=3.0) 